In [23]:
import pandas as pd
import numpy as np
sales= pd.read_csv("sales_date.csv")
sales.dropna(inplace= True)
sales.isnull().sum()

order_id                0
customer_id             0
product_id              0
product_count           0
product_price           0
product_discount        0
price_after_discount    0
order_date              0
dtype: int64

In [24]:
sales.shape

(1494806, 8)

In [25]:
sales.groupby('customer_id').size().sort_values(ascending=False).reset_index(name='order_count').head()

,customer_id,order_count
0,40880,36
1,24271,32
2,94904,32
3,97967,32
4,72034,32


# Feature Creation Cus*order_date ---------- Product ------  

In [26]:
df = sales.sample(100)#[sales["customer_id"] == 24271].copy()
df["order_date"] = pd.to_datetime(df["order_date"])
df["day_of_week"] = df["order_date"].dt.dayofweek
df["day_of_month"] = df["order_date"].dt.day
df["month"] = df["order_date"].dt.month
df["quarter"] = df["order_date"].dt.quarter
df["week_of_year"] = df["order_date"].dt.isocalendar().week
df["is_weekend"] = df["day_of_week"].isin([5, 6]).astype(int)

In [27]:
df['spend'] =df['price_after_discount']*df['product_count']
df.head()

,order_id,customer_id,product_id,product_count,product_price,product_discount,price_after_discount,order_date,day_of_week,day_of_month,month,quarter,week_of_year,is_weekend,spend
381021,1704477,56953,235,15,34.5061,20,27.60,2026-04-28,1,28,4,2,18,0,414.00
254967,1139967,87978,442,23,72.9041,23,56.14,2026-02-18,2,18,2,1,8,0,1291.22
51895,232255,2593,94,1,43.5109,20,34.81,2026-01-09,4,9,1,1,2,0,34.81
696777,3115746,3665,226,1,92.0872,16,77.35,2026-02-24,1,24,2,1,9,0,77.35
503650,2253515,68570,192,18,29.5923,20,23.67,2026-01-04,6,4,1,1,1,1,426.06


In [28]:
df.shape

(100, 15)

In [29]:
cpd = (
    df.groupby(
        ["customer_id", "order_date"],
        as_index=False
    )
    .agg(
        purchase_count=("order_id", "count"),

        # Quantity
        avg_quantity=("product_count", "mean"),
        max_quantity=("product_count", "max"),
        p25_quantity=("product_count", lambda x: x.quantile(0.25)),
        p50_quantity=("product_count", lambda x: x.quantile(0.50)),
        p75_quantity=("product_count", lambda x: x.quantile(0.75)),
        p90_quantity=("product_count", lambda x: x.quantile(0.90)),
        p95_quantity=("product_count", lambda x: x.quantile(0.95)),

        # Spend
        total_spend=("spend", "sum"),
        avg_spend=("spend", "mean"),
        min_spend=("spend", "min"),
        max_spend=("spend", "max"),
        p25_spend=("spend", lambda x: x.quantile(0.25)),
        p50_spend=("spend", lambda x: x.quantile(0.50)),
        p75_spend=("spend", lambda x: x.quantile(0.75)),
        p90_spend=("spend", lambda x: x.quantile(0.90)),
        p95_spend=("spend", lambda x: x.quantile(0.95)),

        # Price after discount
        total_pad=("price_after_discount", "sum"),
        avg_pad=("price_after_discount", "mean"),
        min_pad=("price_after_discount", "min"),
        max_pad=("price_after_discount", "max"),
        p25_pad=("price_after_discount", lambda x: x.quantile(0.25)),
        p50_pad=("price_after_discount", lambda x: x.quantile(0.50)),
        p75_pad=("price_after_discount", lambda x: x.quantile(0.75)),
        p90_pad=("price_after_discount", lambda x: x.quantile(0.90)),
        p95_pad=("price_after_discount", lambda x: x.quantile(0.95)),

        # Product count
        total_pc=("product_count", "sum"),
        avg_pc=("product_count", "mean"),
        min_pc=("product_count", "min"),
        max_pc=("product_count", "max"),
        p25_pc=("product_count", lambda x: x.quantile(0.25)),
        p50_pc=("product_count", lambda x: x.quantile(0.50)),
        p75_pc=("product_count", lambda x: x.quantile(0.75)),
        p90_pc=("product_count", lambda x: x.quantile(0.90)),
        p95_pc=("product_count", lambda x: x.quantile(0.95)),

        # Product discount
        total_pd=("product_discount", "sum"),
        avg_pd=("product_discount", "mean"),
        min_pd=("product_discount", "min"),
        max_pd=("product_discount", "max"),
        p25_pd=("product_discount", lambda x: x.quantile(0.25)),
        p50_pd=("product_discount", lambda x: x.quantile(0.50)),
        p75_pd=("product_discount", lambda x: x.quantile(0.75)),
        p90_pd=("product_discount", lambda x: x.quantile(0.90)),
        p95_pd=("product_discount", lambda x: x.quantile(0.95)),
        
        # Product price
        total_pp=("product_price", "sum"),
        avg_pp=("product_price", "mean"),
        min_pp=("product_price", "min"),
        max_pp=("product_price", "max"),
        p25_pp=("product_price", lambda x: x.quantile(0.25)),
        p50_pp=("product_price", lambda x: x.quantile(0.50)),
        p75_pp=("product_price", lambda x: x.quantile(0.75)),
        p90_pp=("product_price", lambda x: x.quantile(0.90)),
        p95_pp=("product_price", lambda x: x.quantile(0.95)),
        
        mode_product_id=("product_id", lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan),
    )
)

In [30]:
columns = cpd.columns[3:]

print("=" * 100)
print(f"Total columns: {(cpd.shape)}")
print("=" * 100)
for i in range(0, len(columns), 10):
    print(columns[i:i+10].tolist())

Total columns: (100, 56)
['avg_quantity', 'max_quantity', 'p25_quantity', 'p50_quantity', 'p75_quantity', 'p90_quantity', 'p95_quantity', 'total_spend', 'avg_spend', 'min_spend']
['max_spend', 'p25_spend', 'p50_spend', 'p75_spend', 'p90_spend', 'p95_spend', 'total_pad', 'avg_pad', 'min_pad', 'max_pad']
['p25_pad', 'p50_pad', 'p75_pad', 'p90_pad', 'p95_pad', 'total_pc', 'avg_pc', 'min_pc', 'max_pc', 'p25_pc']
['p50_pc', 'p75_pc', 'p90_pc', 'p95_pc', 'total_pd', 'avg_pd', 'min_pd', 'max_pd', 'p25_pd', 'p50_pd']
['p75_pd', 'p90_pd', 'p95_pd', 'total_pp', 'avg_pp', 'min_pp', 'max_pp', 'p25_pp', 'p50_pp', 'p75_pp']
['p90_pp', 'p95_pp', 'mode_product_id']


In [31]:
cpd = cpd.sort_values( ["customer_id", "order_date"]).copy()
for i in cpd.columns:
    cpd[f"last_{i}"] = (cpd.groupby("customer_id")[i] .shift(1) )

In [32]:
columns = cpd.columns[3:]

print("=" * 100)
print(f"Total columns: {(cpd.shape)}")
print("=" * 100)
for i in range(0, len(columns), 10):
    print(columns[i:i+10].tolist())

Total columns: (100, 112)
['avg_quantity', 'max_quantity', 'p25_quantity', 'p50_quantity', 'p75_quantity', 'p90_quantity', 'p95_quantity', 'total_spend', 'avg_spend', 'min_spend']
['max_spend', 'p25_spend', 'p50_spend', 'p75_spend', 'p90_spend', 'p95_spend', 'total_pad', 'avg_pad', 'min_pad', 'max_pad']
['p25_pad', 'p50_pad', 'p75_pad', 'p90_pad', 'p95_pad', 'total_pc', 'avg_pc', 'min_pc', 'max_pc', 'p25_pc']
['p50_pc', 'p75_pc', 'p90_pc', 'p95_pc', 'total_pd', 'avg_pd', 'min_pd', 'max_pd', 'p25_pd', 'p50_pd']
['p75_pd', 'p90_pd', 'p95_pd', 'total_pp', 'avg_pp', 'min_pp', 'max_pp', 'p25_pp', 'p50_pp', 'p75_pp']
['p90_pp', 'p95_pp', 'mode_product_id', 'last_customer_id', 'last_order_date', 'last_purchase_count', 'last_avg_quantity', 'last_max_quantity', 'last_p25_quantity', 'last_p50_quantity']
['last_p75_quantity', 'last_p90_quantity', 'last_p95_quantity', 'last_total_spend', 'last_avg_spend', 'last_min_spend', 'last_max_spend', 'last_p25_spend', 'last_p50_spend', 'last_p75_spend']
['l

In [33]:
cpd.head()

,customer_id,order_date,purchase_count,avg_quantity,max_quantity,p25_quantity,p50_quantity,p75_quantity,p90_quantity,p95_quantity,...,last_total_pp,last_avg_pp,last_min_pp,last_max_pp,last_p25_pp,last_p50_pp,last_p75_pp,last_p90_pp,last_p95_pp,last_mode_product_id
0,804,2026-04-21,1,1.0,1,1.0,1.0,1.0,1.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1079,2026-01-12,1,1.0,1,1.0,1.0,1.0,1.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1325,2026-01-18,1,1.0,1,1.0,1.0,1.0,1.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1987,2026-02-01,1,1.0,1,1.0,1.0,1.0,1.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2593,2026-01-09,1,1.0,1,1.0,1.0,1.0,1.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [34]:
cpd.columns

Index(['customer_id', 'order_date', 'purchase_count', 'avg_quantity',
       'max_quantity', 'p25_quantity', 'p50_quantity', 'p75_quantity',
       'p90_quantity', 'p95_quantity',
       ...
       'last_total_pp', 'last_avg_pp', 'last_min_pp', 'last_max_pp',
       'last_p25_pp', 'last_p50_pp', 'last_p75_pp', 'last_p90_pp',
       'last_p95_pp', 'last_mode_product_id'],
      dtype='object', length=112)

In [35]:
cpd["Same_order"] = (cpd["mode_product_id"] == cpd["last_mode_product_id"]).astype(int)
cpd["Last_transaction_days"] = (cpd["order_date"] - cpd["last_order_date"] ).dt.days
cpd.drop(['last_customer_id','last_mode_product_id'],axis=1 , inplace = True)

In [36]:
cpd.columns

Index(['customer_id', 'order_date', 'purchase_count', 'avg_quantity',
       'max_quantity', 'p25_quantity', 'p50_quantity', 'p75_quantity',
       'p90_quantity', 'p95_quantity',
       ...
       'last_avg_pp', 'last_min_pp', 'last_max_pp', 'last_p25_pp',
       'last_p50_pp', 'last_p75_pp', 'last_p90_pp', 'last_p95_pp',
       'Same_order', 'Last_transaction_days'],
      dtype='object', length=112)

In [37]:
new_cols = [    'purchase_count', 'avg_quantity', 'max_quantity',
                #'p25_quantity','p50_quantity', 'p75_quantity', 'p90_quantity', 'p95_quantity',
                'total_spend', 'avg_spend', 'min_spend', 'max_spend',
                #'p25_spend','p50_spend', 'p75_spend', 'p90_spend', 'p95_spend',
                'total_pad','avg_pad', 'min_pad', 'max_pad', 
                #'p25_pad', 'p50_pad', 'p75_pad','p90_pad', 'p95_pad',
                'total_pc', 'avg_pc', 'min_pc', 'max_pc',
                #'p25_pc', 'p50_pc', 'p75_pc', 'p90_pc', 'p95_pc', 
                'total_pd', 'avg_pd','min_pd', 'max_pd', 
                #'p25_pd', 'p50_pd', 'p75_pd', 'p90_pd', 'p95_pd',
                'total_pp', 'avg_pp', 'min_pp', 'max_pp',
                #'p25_pp', 'p50_pp', 'p75_pp','p90_pp', 'p95_pp' 
           ]

In [38]:
for i in new_cols : 
    windows = [2, 3, 7, 30]
    group_cols = ["customer_id"]
    # Variables for which you want historical rolling features
    rolling_cols = [ i ]
    
    cpd = cpd.sort_values( group_cols + ["order_date"] ).reset_index(drop=True)
    cpd["order_date"] = pd.to_datetime(cpd["order_date"])
    for col in rolling_cols:
        #print(f"\nProcessing: {col}")
        temp = cpd[ group_cols + ["order_date", col] ].copy()
        temp[col] = ( temp.groupby(group_cols)[col].shift(1) )
        temp = temp.set_index("order_date")
        rolling = (temp.groupby(group_cols)[col].rolling("2D", closed="both"))
    
        for days in windows:
            rolling = (temp.groupby(group_cols)[col].rolling( f"{days}D", closed="both", min_periods=1))
            cpd[f"{col}_{days}d_mean"] = ( rolling.mean() .reset_index(level=group_cols, drop=True) .to_numpy())
            cpd[f"{col}_{days}d_min"]  = ( rolling.min() .reset_index(level=group_cols, drop=True) .to_numpy())
            cpd[f"{col}_{days}d_max"]  = ( rolling.max() .reset_index(level=group_cols, drop=True) .to_numpy())
            cpd[f"{col}_{days}d_std"]  = ( rolling.std() .reset_index(level=group_cols, drop=True) .to_numpy())
            cpd[f"{col}_{days}d_var"]  = ( rolling.var() .reset_index(level=group_cols, drop=True) .to_numpy())
            cpd[f"{col}_{days}d_p25"]  = ( rolling.quantile(0.25) .reset_index(level=group_cols, drop=True) .to_numpy())
            cpd[f"{col}_{days}d_p50"]  = ( rolling.quantile(0.50) .reset_index(level=group_cols, drop=True) .to_numpy())
            cpd[f"{col}_{days}d_p75"]  = ( rolling.quantile(0.75) .reset_index(level=group_cols, drop=True) .to_numpy())
            cpd[f"{col}_{days}d_p90"]  = ( rolling.quantile(0.90) .reset_index(level=group_cols, drop=True) .to_numpy())
            cpd[f"{col}_{days}d_p95"]  = ( rolling.quantile(0.95) .reset_index(level=group_cols, drop=True) .to_numpy())
    
    cpd = cpd.sort_values( group_cols + ["order_date"] ).reset_index(drop=True)
    
    rolling_features = [
        c for c in cpd.columns
        if any( f"_{d}d_" in c for d in windows ) ]
    
    #print("\nNumber of rolling features:", len(rolling_features))
    
#print("\nRolling features:")
#print(rolling_features)

In [39]:
columns = cpd.columns[3:]

print("=" * 100)
print(f"Total columns: {(cpd.shape)}")
print("=" * 100)
for i in range(0, len(columns), 10):
    print(columns[i:i+10].tolist())

Total columns: (100, 1032)
['avg_quantity', 'max_quantity', 'p25_quantity', 'p50_quantity', 'p75_quantity', 'p90_quantity', 'p95_quantity', 'total_spend', 'avg_spend', 'min_spend']
['max_spend', 'p25_spend', 'p50_spend', 'p75_spend', 'p90_spend', 'p95_spend', 'total_pad', 'avg_pad', 'min_pad', 'max_pad']
['p25_pad', 'p50_pad', 'p75_pad', 'p90_pad', 'p95_pad', 'total_pc', 'avg_pc', 'min_pc', 'max_pc', 'p25_pc']
['p50_pc', 'p75_pc', 'p90_pc', 'p95_pc', 'total_pd', 'avg_pd', 'min_pd', 'max_pd', 'p25_pd', 'p50_pd']
['p75_pd', 'p90_pd', 'p95_pd', 'total_pp', 'avg_pp', 'min_pp', 'max_pp', 'p25_pp', 'p50_pp', 'p75_pp']
['p90_pp', 'p95_pp', 'mode_product_id', 'last_order_date', 'last_purchase_count', 'last_avg_quantity', 'last_max_quantity', 'last_p25_quantity', 'last_p50_quantity', 'last_p75_quantity']
['last_p90_quantity', 'last_p95_quantity', 'last_total_spend', 'last_avg_spend', 'last_min_spend', 'last_max_spend', 'last_p25_spend', 'last_p50_spend', 'last_p75_spend', 'last_p90_spend']
['la

In [40]:
cpd.columns

Index(['customer_id', 'order_date', 'purchase_count', 'avg_quantity',
       'max_quantity', 'p25_quantity', 'p50_quantity', 'p75_quantity',
       'p90_quantity', 'p95_quantity',
       ...
       'max_pp_30d_mean', 'max_pp_30d_min', 'max_pp_30d_max', 'max_pp_30d_std',
       'max_pp_30d_var', 'max_pp_30d_p25', 'max_pp_30d_p50', 'max_pp_30d_p75',
       'max_pp_30d_p90', 'max_pp_30d_p95'],
      dtype='object', length=1032)

In [41]:
columns = cpd.columns

print("=" * 100)
print(f"Total columns: {(cpd.shape)}")
print("=" * 10)
for i in range(0, len(columns), 10):
    print(columns[i:i+10].tolist())

Total columns: (100, 1032)
['customer_id', 'order_date', 'purchase_count', 'avg_quantity', 'max_quantity', 'p25_quantity', 'p50_quantity', 'p75_quantity', 'p90_quantity', 'p95_quantity']
['total_spend', 'avg_spend', 'min_spend', 'max_spend', 'p25_spend', 'p50_spend', 'p75_spend', 'p90_spend', 'p95_spend', 'total_pad']
['avg_pad', 'min_pad', 'max_pad', 'p25_pad', 'p50_pad', 'p75_pad', 'p90_pad', 'p95_pad', 'total_pc', 'avg_pc']
['min_pc', 'max_pc', 'p25_pc', 'p50_pc', 'p75_pc', 'p90_pc', 'p95_pc', 'total_pd', 'avg_pd', 'min_pd']
['max_pd', 'p25_pd', 'p50_pd', 'p75_pd', 'p90_pd', 'p95_pd', 'total_pp', 'avg_pp', 'min_pp', 'max_pp']
['p25_pp', 'p50_pp', 'p75_pp', 'p90_pp', 'p95_pp', 'mode_product_id', 'last_order_date', 'last_purchase_count', 'last_avg_quantity', 'last_max_quantity']
['last_p25_quantity', 'last_p50_quantity', 'last_p75_quantity', 'last_p90_quantity', 'last_p95_quantity', 'last_total_spend', 'last_avg_spend', 'last_min_spend', 'last_max_spend', 'last_p25_spend']
['last_p50_

In [42]:
cpd.rename({'mode_product_id':'product_id'},axis=1 , inplace=True)

In [43]:
columns = cpd.columns

print("=" * 100)
print(f"Total columns: {(cpd.shape)}")
print("=" * 10)
for i in range(0, len(columns), 10):
    print(columns[i:i+10].tolist())

Total columns: (100, 1032)
['customer_id', 'order_date', 'purchase_count', 'avg_quantity', 'max_quantity', 'p25_quantity', 'p50_quantity', 'p75_quantity', 'p90_quantity', 'p95_quantity']
['total_spend', 'avg_spend', 'min_spend', 'max_spend', 'p25_spend', 'p50_spend', 'p75_spend', 'p90_spend', 'p95_spend', 'total_pad']
['avg_pad', 'min_pad', 'max_pad', 'p25_pad', 'p50_pad', 'p75_pad', 'p90_pad', 'p95_pad', 'total_pc', 'avg_pc']
['min_pc', 'max_pc', 'p25_pc', 'p50_pc', 'p75_pc', 'p90_pc', 'p95_pc', 'total_pd', 'avg_pd', 'min_pd']
['max_pd', 'p25_pd', 'p50_pd', 'p75_pd', 'p90_pd', 'p95_pd', 'total_pp', 'avg_pp', 'min_pp', 'max_pp']
['p25_pp', 'p50_pp', 'p75_pp', 'p90_pp', 'p95_pp', 'product_id', 'last_order_date', 'last_purchase_count', 'last_avg_quantity', 'last_max_quantity']
['last_p25_quantity', 'last_p50_quantity', 'last_p75_quantity', 'last_p90_quantity', 'last_p95_quantity', 'last_total_spend', 'last_avg_spend', 'last_min_spend', 'last_max_spend', 'last_p25_spend']
['last_p50_spend

In [44]:
cpd.to_csv("Final_Feature_code.csv")